<a href="https://colab.research.google.com/github/yeonji200522-oss/Hands-on-Machine-Learning/blob/main/Py%EB%A1%9C_%EB%B3%80%ED%99%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.neighbors import BallTree
from collections import OrderedDict

# ============================================================
# 1. 거리 및 통계 보조 함수
# ============================================================
def haversine_km_vec(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * (np.sin(dlon / 2.0) ** 2)
    c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    return R * c

def rmse_fitlm(res):
    return float(np.sqrt(res.mse_resid))

def fit_model(df, y_col, x_cols):
    X = sm.add_constant(df[x_cols], has_constant="add")
    y = df[y_col]
    return sm.OLS(y, X, missing="drop").fit()

# ============================================================
# 2. 공간 가중치(WY) 계산 (n×n 효과 반영 로직)
# ============================================================
def make_unique_sum_count(lat, lon, y):
    df_loc = pd.DataFrame({"y": lat, "x": lon, "Y": y})
    # 좌표별 가격 합계와 관측치 개수 계산
    g = df_loc.groupby(["y", "x"], sort=True)["Y"].agg(["sum", "count"]).reset_index()
    g["_uix"] = np.arange(len(g), dtype=int)

    lat_uni = g["y"].to_numpy(dtype=float)
    lon_uni = g["x"].to_numpy(dtype=float)
    sumY_uni = g["sum"].to_numpy(dtype=float)
    count_uni = g["count"].to_numpy(dtype=float)

    # 원본 데이터를 유일 좌표 인덱스에 매핑
    idx_map = df_loc[["y", "x"]].merge(
        g[["y", "x", "_uix"]], on=["y", "x"], how="left", sort=False
    )["_uix"].to_numpy(dtype=int)

    return lat_uni, lon_uni, sumY_uni, count_uni, idx_map

def compute_WY_logic(lat_uni, lon_uni, sumY_uni, count_uni, distance_band_km=1.0):
    lat_r, lon_r = np.deg2rad(lat_uni), np.deg2rad(lon_uni)
    coords = np.column_stack([lat_r, lon_r])

    tree = BallTree(coords, metric="haversine")
    n = len(lat_uni)
    WY_uni = np.zeros(n)
    rad_band = (distance_band_km / 6371.0) * (1.0 + 1e-12)

    for i in range(n):
        idx = tree.query_radius(coords[i:i+1], r=rad_band)[0]
        idx = idx[idx != i] # 자기 자신 제외

        if idx.size > 0:
            d_km = haversine_km_vec(lat_r[i], lon_r[i], lat_r[idx], lon_r[idx])
            mask = (d_km <= distance_band_km + 1e-12) & (d_km > 0)
            if np.any(mask):
                d_sub, sY_sub, c_sub = d_km[mask], sumY_uni[idx[mask]], count_uni[idx[mask]]
                inv_d = 1.0 / d_sub
                # 핵심: (가중치 * 가격합) / (가중치 * 관측치수) -> n*n 효과 재현
                WY_uni[i] = np.sum(inv_d * sY_sub) / np.sum(inv_d * c_sub)
    return WY_uni

# ============================================================
# 3. 포맷팅 및 테이블 생성
# ============================================================
def mark_1pct(res, var, digits=3):
    if var not in res.params.index: return "–"
    coef, pval = res.params[var], res.pvalues[var]
    return f"{coef:.{digits}f}" + ("‡" if pval < 0.01 else "")

def fmt_sci_mark(res, var):
    if var not in res.params.index: return "–"
    coef, pval = res.params[var], res.pvalues[var]
    exp = int(np.floor(np.log10(abs(coef))))
    if exp <= -4:
        mant = coef / (10 ** exp)
        s = f"{mant:.1f} × 10$^{{{exp}}}$"
    else: s = f"{coef:.3f}"
    return s + ("‡" if pval < 0.01 else "")

def build_formatted_table(results_dict, row_map):
    cols = list(results_dict.keys())
    T = pd.DataFrame(index=list(row_map.keys()), columns=cols)
    for col in cols:
        res = results_dict[col]
        for rname, vname in row_map.items():
            if vname is None: T.loc[rname, col] = ""
            elif rname in ["Units", "Pop. density"]: T.loc[rname, col] = fmt_sci_mark(res, vname)
            else: T.loc[rname, col] = mark_1pct(res, vname)

        T.loc["F-statistics", col] = f"{round(res.fvalue, -1):,.0f}‡"
        T.loc["RMSE", col] = f"{rmse_fitlm(res):.3f}"
        T.loc["Adjusted ", col] = f"{res.rsquared_adj:.3f}"
    return T

# ============================================================
# 4. 실행 메인
# ============================================================
file_name = '0714_busan201819.xlsx' # 또는 .csv 파일명
df = pd.read_excel(file_name) if file_name.endswith('xlsx') else pd.read_csv(file_name)

# 1) WY 계산
lat_uni, lon_uni, sumY_uni, count_uni, idx_map = make_unique_sum_count(df['y'], df['x'], df['Price'])
WY_uni = compute_WY_logic(lat_uni, lon_uni, sumY_uni, count_uni)
df["IND_spw"] = WY_uni[idx_map]

# 2) 변수 정의
base = ["Area","Floor","Households","Parking","Heating","Year","Dist. Subway","Top Univ.","Sex ratio","Pop. Density","Higher degree","Medium age","Spring","Fall","Winter"]
x1, x2, x3 = base+["Dist. Green","Dist. Water"], base+["Dist. Water","HGVI_50"], base+["Dist. Green","Dist. Water","HGVI_50"]

# 3) 최적 Rho 탐색
best_rho, best_rmse = 0, float('inf')
for rho in np.round(np.arange(-0.99, 1.00, 0.01), 2):
    df["_tmp_y"] = df["Price"] - rho * df["IND_spw"]
    res = fit_model(df, "_tmp_y", x3)
    if rmse_fitlm(res) < best_rmse:
        best_rmse, best_rho = rmse_fitlm(res), rho

# 4) 결과 테이블 생성
df["Y_splag"] = df["Price"] - best_rho * df["IND_spw"]
res_ols = {"(1)": fit_model(df, "Price", x1), "(2)": fit_model(df, "Price", x2), "(3)": fit_model(df, "Price", x3)}
res_lag = {"(1)": fit_model(df, "Y_splag", x1), "(2)": fit_model(df, "Y_splag", x2), "(3)": fit_model(df, "Y_splag", x3)}

row_map = OrderedDict([
    ("Property characteristics", None), ("Size", "Area"), ("Floor", "Floor"), ("Units", "Households"),
    ("Parking", "Parking"), ("Heating", "Heating"), ("Year", "Year"),
    ("Environmental amenities", None), ("Dist. green", "Dist. Green"), ("Dist. water", "Dist. Water"), ("Green index", "HGVI_50"),
    ("Local built environment", None), ("Dist. subway", "Dist. Subway"), ("Top univ.", "Top Univ."),
    ("Local demographics", None), ("Sex ratio", "Sex ratio"), ("Pop. density", "Pop. Density"), ("Higher degree", "Higher degree"), ("Medium age", "Medium age"),
    ("Seasonality control", None), ("Spring", "Spring"), ("Fall", "Fall"), ("Winter", "Winter"),
    ("F-statistics", "F"), ("RMSE", "RMSE"), ("Adjusted ", "AdjR2")
])

print(f"전체 관측치 = {len(df):,} {len(lat_uni):,}")
print(f"중복 제거 관측치 = {len(lat_uni):,}")
print(f"Best rho = {best_rho}")
print(f"Net RMSE = {best_rmse}")
print(f"OLS RMSE = {rmse_fitlm(res_ols['(3)'])}")
print("\nTable 4 (OLS): Obs.= {:,}".format(len(df)))
print(build_formatted_table(res_ols, row_map))
print("\nTable 5 (Spatial lag): Obs.= {:,}".format(len(df)))
print(build_formatted_table(res_lag, row_map))

전체 관측치 = 52,644 2,404
중복 제거 관측치 = 2,404
Best rho = 0.24
Net RMSE = 0.29195369517026654
OLS RMSE = 0.314679347751799

Table 4 (OLS): Obs.= 52,644
                                       (1)               (2)               (3)
Property characteristics                                                      
Size                                0.012‡            0.012‡            0.012‡
Floor                               0.005‡            0.005‡            0.005‡
Units                     8.4 × 10$^{-5}$‡  8.6 × 10$^{-5}$‡  8.5 × 10$^{-5}$‡
Parking                             0.103‡            0.103‡            0.102‡
Heating                             0.160‡            0.158‡            0.158‡
Year                                0.013‡            0.013‡            0.013‡
Environmental amenities                                                       
Dist. green                        -0.008‡                 –           -0.008‡
Dist. water                        -0.012‡           -0.011‡     

In [ ]:
vif_vars = [
    "Area", "Floor", "Households", "Parking", "Heating", "Year",
    "Dist. Green", "Dist. Water", "HGVI_50", "Dist. Subway", "Top Univ.",
    "Sex ratio", "Pop. Density", "Higher degree", "Medium age",
    "Spring", "Fall", "Winter"
]

# 3. VIF 계산 로직 (매틀랩 방식: 상관행렬의 역행렬 대각요소)
# X_vif 준비
X_vif = df[vif_vars].values

# 상관계수 행렬 계산 (R0 = corrcoef(table2array(tbl_vif)))
R0 = np.corrcoef(X_vif, rowvar=False)

# 역행렬의 대각 성분 추출 (V = diag(inv(R0))')
V = np.diag(np.linalg.inv(R0))

# 4. 결과 출력 (변수명과 매칭)
print("--- V (VIF) Results ---")
vif_result = pd.Series(V, index=vif_vars)
print(vif_result.round(4)) # 소수점 4자리까지 출력

--- V (VIF) Results ---
Area             1.2358
Floor            1.1570
Households       1.1387
Parking          1.2238
Heating          1.2220
Year             1.1511
Dist. Green      1.0739
Dist. Water      1.0766
HGVI_50          1.0555
Dist. Subway     1.1895
Top Univ.        1.1516
Sex ratio        1.4633
Pop. Density     1.1570
Higher degree    1.7915
Medium age       1.5797
Spring           1.6468
Fall             1.8257
Winter           1.6865
dtype: float64


###걍 테스트


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.neighbors import BallTree
from collections import OrderedDict

# ============================================================
# 1. 공간 가중치(WY) 계산
# ============================================================
def compute_spatial_weights(df, distance_band_km=1.0):
    # 중복 좌표별 통계 계산
    loc_stats = df.groupby(['y', 'x'])['Price'].agg(['sum', 'count']).reset_index()
    lat_uni = np.deg2rad(loc_stats['y'].values)
    lon_uni = np.deg2rad(loc_stats['x'].values)
    sumY_uni = loc_stats['sum'].values
    count_uni = loc_stats['count'].values

    coords = np.column_stack([lat_uni, lon_uni])
    tree = BallTree(coords, metric='haversine')

    n_uni = len(loc_stats)
    WY_uni = np.zeros(n_uni)
    radius = distance_band_km / 6371.0 + 1e-12

    for i in range(n_uni):
        idx = tree.query_radius(coords[i:i+1], r=radius)[0]
        idx = idx[idx != i] # 자기 자신 제외

        if len(idx) > 0:
            # 하버사인 거리 계산
            dlat = lat_uni[idx] - lat_uni[i]
            dlon = lon_uni[idx] - lon_uni[i]
            a = np.sin(dlat/2)**2 + np.cos(lat_uni[i]) * np.cos(lat_uni[idx]) * np.sin(dlon/2)**2
            d_km = 6371.0 * 2 * np.arcsin(np.sqrt(a))

            # 가중치 w = 1/d 적용 및 n*n 효과 재현
            weights = 1.0 / d_km
            WY_uni[i] = np.sum(weights * sumY_uni[idx]) / np.sum(weights * count_uni[idx])

    # 전체 데이터(52,644행)에 매핑
    loc_stats['WY_val'] = WY_uni
    df_merged = df.merge(loc_stats[['y', 'x', 'WY_val']], on=['y', 'x'], how='left')
    return df_merged['WY_val'].values, n_uni

# ============================================================
# 2. 결과 포맷팅 및 VIF 계산 함수
# ============================================================
def fmt_academic(res, var):
    if var not in res.params.index: return " – "
    coef, pval = res.params[var], res.pvalues[var]
    mark = "‡" if pval < 0.01 else ""

    # Households(Units)와 Pop. Density는 지수 표기법 적용 (공백 유지)
    if var in ["Households", "Pop. Density"]:
        exp = int(np.floor(np.log10(abs(coef))))
        mant = coef / (10 ** exp)
        return f"{mant:.1f} × 10$^{{{exp}}}${mark}"
    return f"{coef:.3f}{mark}"

def build_formatted_table(results_dict, row_map):
    cols = list(results_dict.keys())
    T = pd.DataFrame(index=list(row_map.keys()), columns=cols)
    for col in cols:
        res = results_dict[col]
        for rname, vname in row_map.items():
            if vname is None: T.loc[rname, col] = ""
            else: T.loc[rname, col] = fmt_academic(res, vname)

        # 하단 요약 지표 (F값 10단위 반올림 및 콤마 적용)
        T.loc["F-statistics", col] = f"{int(round(res.fvalue, -1)):,d}‡"
        T.loc["RMSE", col] = f"{np.sqrt(res.mse_resid):.3f}"
        T.loc["Adjusted", col] = f"{res.rsquared_adj:.3f}"
    return T

# ============================================================
# 3. 메인 분석 실행
# ============================================================
# 데이터 로드
df = pd.read_excel('0714_busan201819.xlsx')

# 공간 가중치 변수 생성
df['IND_spw'], n_uni = compute_spatial_weights(df)

# 변수 구성 정의
base = ["Area", "Floor", "Households", "Parking", "Heating", "Year", "Dist. Subway", "Top Univ.",
        "Sex ratio", "Pop. Density", "Higher degree", "Medium age", "Spring", "Fall", "Winter"]
m_vars = {
    "(1)": base + ["Dist. Green", "Dist. Water"],
    "(2)": base + ["Dist. Water", "HGVI_50"],
    "(3)": base + ["Dist. Green", "Dist. Water", "HGVI_50"]
}

# 모델 피팅 (Best rho = 0.24 적용)
best_rho = 0.24
df["Y_splag"] = df["Price"] - best_rho * df["IND_spw"]

res_ols = {k: sm.OLS(df['Price'], sm.add_constant(df[v])).fit() for k, v in m_vars.items()}
res_lag = {k: sm.OLS(df['Y_splag'], sm.add_constant(df[v])).fit() for k, v in m_vars.items()}

# VIF 계산 (Model 3 기준)
X_vif = df[m_vars["(3)"]].values
R0 = np.corrcoef(X_vif, rowvar=False)
VIF_values = np.diag(np.linalg.inv(R0))
vif_series = pd.Series(VIF_values, index=m_vars["(3)"])

# 출력용 행 매핑
row_map = OrderedDict([
    ("Property characteristics", None), ("Size", "Area"), ("Floor", "Floor"), ("Units", "Households"),
    ("Parking", "Parking"), ("Heating", "Heating"), ("Year", "Year"),
    ("Environmental amenities", None), ("Dist. green", "Dist. Green"), ("Dist. water", "Dist. Water"), ("Green index", "HGVI_50"),
    ("Local built environment", None), ("Dist. subway", "Dist. Subway"), ("Top univ.", "Top Univ."),
    ("Local demographics", None), ("Sex ratio", "Sex ratio"), ("Pop. density", "Pop. Density"), ("Higher degree", "Higher degree"), ("Medium age", "Medium age"),
    ("Seasonality control", None), ("Spring", "Spring"), ("Fall", "Fall"), ("Winter", "Winter"),
    ("F-statistics", "F"), ("RMSE", "RMSE"), ("Adjusted", "AdjR2")
])

# ============================================================
# 4. 결과 출력
# ============================================================
print(f"전체 관측치 = {len(df):,} {n_uni:,}")
print(f"중복 제거 관측치 = {n_uni:,}")
print(f"Best rho = {best_rho}")
print(f"Net RMSE = {np.sqrt(res_lag['(3)'].mse_resid):.16f}")
print(f"OLS RMSE = {np.sqrt(res_ols['(3)'].mse_resid):.15f}")

print("\nTable 4 (OLS): Obs.= {:,}".format(len(df)))
print(build_formatted_table(res_ols, row_map))

print("\nTable 5 (Spatial lag): Obs.= {:,}".format(len(df)))
print(build_formatted_table(res_lag, row_map))

print("\n--- V (VIF) Results (Model 3) ---")
print(vif_series.round(4))

전체 관측치 = 52,644 2,404
중복 제거 관측치 = 2,404
Best rho = 0.24
Net RMSE = 0.2919536951702666
OLS RMSE = 0.314679347751799

Table 4 (OLS): Obs.= 52,644
                                       (1)               (2)               (3)
Property characteristics                                                      
Size                                0.012‡            0.012‡            0.012‡
Floor                               0.005‡            0.005‡            0.005‡
Units                     8.4 × 10$^{-5}$‡  8.6 × 10$^{-5}$‡  8.5 × 10$^{-5}$‡
Parking                             0.103‡            0.103‡            0.102‡
Heating                             0.160‡            0.158‡            0.158‡
Year                                0.013‡            0.013‡            0.013‡
Environmental amenities                                                       
Dist. green                        -0.008‡                –            -0.008‡
Dist. water                        -0.012‡           -0.011‡      